> **Note:** This notebook is an early end-to-end CNN experiment. Training collapsed to predicting a single class, which led to the final design.
>
> The final ultrasound pipeline (EfficientNet-B0 embeddings + LBP texture features → CatBoost, and the ResNet50 Grad-CAM model) lives in `scripts/` — see `train_ultrasound_catboost.py`, `train_resnet50_gradcam.py` and the README for results.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

print(tf.__version__)


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)


In [ ]:
train_generator = train_datagen.flow_from_directory(
    "../data/ultrasound/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_generator = test_datagen.flow_from_directory(
    "../data/ultrasound/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

train_generator.class_indices


In [ ]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

model.summary()


In [ ]:
import os
from PIL import Image

def remove_corrupt_images(base_dir):
    removed = 0
    for root, _, files in os.walk(base_dir):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                with Image.open(file_path) as img:
                    img.verify()  # Verify image integrity
            except Exception:
                print("Removing corrupt image:", file_path)
                os.remove(file_path)
                removed += 1
    print(f"Total removed images: {removed}")


In [ ]:
remove_corrupt_images("../data/ultrasound/train")
remove_corrupt_images("../data/ultrasound/test")


In [ ]:
EPOCHS = 15

history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=EPOCHS
)


In [ ]:
for layer in base_model.layers[-20:]:
    layer.trainable = True


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)


In [ ]:
FINE_TUNE_EPOCHS = 10

history_fine = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=FINE_TUNE_EPOCHS
)


In [ ]:
test_generator.reset()

y_prob = model.predict(test_generator).ravel()
y_true = test_generator.classes

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

print("Test ROC-AUC:", roc_auc_score(y_true, y_prob))
print(confusion_matrix(y_true, (y_prob >= 0.5).astype(int)))
print(classification_report(y_true, (y_prob >= 0.5).astype(int)))


In [ ]:
# Invert probabilities so that PCOS = 1
y_prob_fixed = 1 - y_prob

from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

print("Fixed Test ROC-AUC:", roc_auc_score(y_true, y_prob_fixed))

y_pred_fixed = (y_prob_fixed >= 0.5).astype(int)

print(confusion_matrix(y_true, y_pred_fixed))
print(classification_report(y_true, y_pred_fixed))
